# RL Activation Steering for Hallucination Correction
## TriviaQA — Everything from scratch

---

## The idea

When a language model hallucinates, its internal hidden state at a critical layer drifts toward a *hallucination region* in activation space. This notebook trains an RL policy to detect that drift and push the activation back toward the *factual region*.

```
h at L*  →  SteeringPolicy  →  Δh
h + Δh  is closer to factual centroid  →  +reward
h + Δh  is farther from factual centroid  →  -reward
```

This is **correction**, not just detection.

---

## What this notebook does

| Cell | What runs | Time |
|---|---|---|
| 0 | Install packages | 1 min |
| 1 | Load Llama-3.2-3B (frozen) | 3 min |
| 2 | Generate TriviaQA responses + labels | ~36 min |
| 3 | Find critical layer L* automatically | ~10 min |
| 4 | Early detection sweep (10%-100% tokens) | ~5 min |
| 5 | Define RL steering policy | instant |
| 6 | Train RL steering policy | ~15 min |
| 7 | Evaluate before vs after steering | ~5 min |
| 8 | 5 figures + final results table | ~2 min |

**Total: ~90 minutes on A100**

---

## Before running

1. Replace `your_hf_token_here` in **Cell 1** with your HuggingFace token
2. Runtime → Change runtime type → **A100 GPU**
3. Runtime → **Run all**

> All results saved to `/content/drive/MyDrive/facr_triviaqa/`

### CELL 0 — Install dependencies

Run this first. If prompted to restart runtime, do so,
then run all cells again from Cell 1.

In [ ]:
# ── CELL 0 — Install dependencies ────────────────────────────────
# Run this first. If prompted to restart runtime, do so,
# then run all cells again from Cell 1.

# !pip install transformers datasets accelerate tqdm scikit-learn matplotlib -q


### CELL 1 — Imports + frozen Llama-3.2-3B loading

Loads all libraries and the frozen Llama-3.2-3B-Instruct model.
The model is loaded with output_hidden_states=True so we can
extract internal activations at any layer.
IMPORTANT: Replace "your_hf_token_here" below with your token.
Get one at: https://huggingface.co/settings/tokens
Accept Llama-3.2 license at:
https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

In [ ]:
# ── CELL 1 — Imports and model loading ───────────────────────────
#
# Loads all libraries and the frozen Llama-3.2-3B-Instruct model.
# The model is loaded with output_hidden_states=True so we can
# extract internal activations at any layer.
#
# IMPORTANT: Replace "your_hf_token_here" below with your token.
# Get one at: https://huggingface.co/settings/tokens
# Accept Llama-3.2 license at:
# https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

import os, json, re, string, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    pass  # already mounted

# all results saved here
SAVE_DIR = '/content/drive/MyDrive/facr_triviaqa'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

from huggingface_hub import login
login("<REDACTED>")  # <── REPLACE THIS

MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE      = "cuda"
NUM_LAYERS  = 28      # 0=embedding, 1-28=transformer blocks
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"\nDevice : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,
)
model.eval()
print(f"Model  : {MODEL_ID} ✓")
print(f"Layers : {NUM_LAYERS + 1} "
      f"(0=embedding, 1-{NUM_LAYERS}=transformer blocks)")


### CELL 2 — Generate TriviaQA data from scratch (~36 min)

Generates responses for 500 TriviaQA questions (4 per question
= 2000 total samples) and labels each with token F1 against
the gold answer.
factuality_score > 0.5 → FACTUAL
factuality_score ≤ 0.5 → HALLUCINATED
Always regenerates from scratch (no caching).
Checkpoints every 100 questions so if Colab disconnects
you can resume from the last checkpoint.
Expected runtime: ~36 minutes on A100.

In [ ]:
# ── CELL 2 — Generate TriviaQA data ──────────────────────────────
#
# Generates responses for 500 TriviaQA questions (4 per question
# = 2000 total samples) and labels each with token F1 against
# the gold answer.
#
# factuality_score > 0.5 → FACTUAL
# factuality_score ≤ 0.5 → HALLUCINATED
#
# Always regenerates from scratch (no caching).
# Checkpoints every 100 questions so if Colab disconnects
# you can resume from the last checkpoint.
#
# Expected runtime: ~36 minutes on A100.

def normalize(text):
    """Normalize text for F1 scoring."""
    text = text.lower().strip()
    text = re.sub(r'\b(1\d{2}|\d{2})0s?\b',
                  lambda m: m.group(0).rstrip('s'), text)
    text = re.sub(r'\b(the|a|an)\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def token_f1(pred, ref):
    """Token overlap F1 between prediction and reference."""
    p_tok = normalize(pred).split()
    r_tok = normalize(ref).split()
    if not p_tok or not r_tok: return 0.0
    common = set(p_tok) & set(r_tok)
    if not common: return 0.0
    p = len(common) / len(p_tok)
    r = len(common) / len(r_tok)
    return 2 * p * r / (p + r)

def generate_response(question):
    """Generate a short factual response from Llama."""
    prompt = (
        "Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=200
    ).to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=40,
            temperature=0.8, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(
        new_tokens, skip_special_tokens=True).strip()
    return decoded.split('\n')[0].split('.')[0].strip()

print("Generating TriviaQA responses from scratch...")
print("This will take ~36 minutes. "
      "Checkpointed every 100 questions.\n")

dataset      = load_dataset(
    "trivia_qa", "rc.nocontext", split="train[:500]")
labeled_data = []
dist         = {'factual': 0, 'hallu': 0}

for i, example in enumerate(tqdm(dataset)):
    question   = example['question']
    ans        = example['answer']
    prompt     = (
        "Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\nAnswer:"
    )

    # collect all valid gold answers
    candidates = []
    if ans.get('value'):
        candidates.append(ans['value'])
    for a in ans.get('aliases', []):
        if isinstance(a, str): candidates.append(a)
    for a in ans.get('normalized_aliases', []):
        if isinstance(a, str): candidates.append(a)

    for _ in range(4):
        resp  = generate_response(question)
        score = max((token_f1(resp, ref)
                     for ref in candidates), default=0.0)
        labeled_data.append({
            'question':         question,
            'response':         resp,
            'factuality_score': float(score),
            'prompt':           prompt
        })
        if score > 0.5: dist['factual'] += 1
        else:           dist['hallu']   += 1

    # checkpoint every 100 questions
    if (i + 1) % 100 == 0:
        ckpt = f'{SAVE_DIR}/labeled_data_ckpt_{i+1}.json'
        with open(ckpt, 'w') as f:
            json.dump(labeled_data, f)
        print(f"  Checkpoint: {i+1}/500 questions | "
              f"{len(labeled_data)} samples | "
              f"factual={dist['factual']} "
              f"hallu={dist['hallu']}")

# save final
data_path = f'{SAVE_DIR}/labeled_data.json'
with open(data_path, 'w') as f:
    json.dump(labeled_data, f)

total = len(labeled_data)
print(f"\nGeneration complete!")
print(f"Total samples       : {total}")
print(f"Factual  (>0.5)     : {dist['factual']} "
      f"({100*dist['factual']/total:.1f}%)")
print(f"Hallucinated (≤0.5) : {dist['hallu']} "
      f"({100*dist['hallu']/total:.1f}%)")
print(f"Saved to: {data_path}")

# fixed train/eval split (80/20)
random.seed(RANDOM_SEED)
random.shuffle(labeled_data)
TRAIN_DATA = labeled_data[:1600]
EVAL_DATA  = labeled_data[1600:]
print(f"\nTrain: {len(TRAIN_DATA)} | Eval: {len(EVAL_DATA)}")


### CELL 3 — Find critical layer L* automatically

Sweeps all 28 layers of the frozen model and for each layer
trains a linear probe to separate factual from hallucinated
hidden states.
L* = the layer where this separation is strongest (peak AUROC).
THREE-WAY SPLIT prevents data leakage:
20% of train → layer selection (find L*)
60% of train → probe training  (train at L*)
20% of train → not used here  (evaluation in Cell 4)
This answers: WHERE does the model "know" it's hallucinating?

In [ ]:
# ── CELL 3 — Find critical layer L* automatically ────────────────
#
# Sweeps all 28 layers of the frozen model and for each layer
# trains a linear probe to separate factual from hallucinated
# hidden states.
#
# L* = the layer where this separation is strongest (peak AUROC).
#
# THREE-WAY SPLIT prevents data leakage:
#   20% of train → layer selection (find L*)
#   60% of train → probe training  (train at L*)
#   20% of train → not used here  (evaluation in Cell 4)
#
# This answers: WHERE does the model "know" it's hallucinating?

def extract_hidden_states(data, layer_idx, cutoff_pct=1.0):
    """
    Extract last-token hidden state at layer_idx for each sample.

    cutoff_pct=1.0 → full response
    cutoff_pct=0.2 → first 20% of tokens (early detection)

    Returns:
        hiddens (N, 3072) float32 cpu tensor
        labels  (N,)      1=factual, 0=hallucinated
    """
    hiddens, labels = [], []
    for item in data:
        full_text = item['prompt'] + " " + item['response']
        inputs    = tokenizer(
            full_text, return_tensors='pt',
            truncation=True, max_length=256
        ).to(DEVICE)
        if cutoff_pct < 1.0:
            total  = inputs['input_ids'].shape[1]
            cutoff = max(4, int(total * cutoff_pct))
            inputs = {k: v[:, :cutoff] for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
        h = out.hidden_states[layer_idx][0, -1, :].float().cpu()
        hiddens.append(h)
        labels.append(1 if item['factuality_score'] > 0.5 else 0)
    return torch.stack(hiddens), torch.tensor(labels)

def quick_auroc(h, l, cv=3):
    """Fast 3-fold AUROC for layer selection."""
    X = h.numpy(); y = l.numpy()
    if len(np.unique(y)) < 2: return 0.5
    return float(cross_val_score(
        LogisticRegression(max_iter=300),
        X, y, cv=cv, scoring='roc_auc').mean())

def robust_auroc(h, l, cv=10):
    """Rigorous 10-fold AUROC for final evaluation."""
    X   = h.numpy(); y = l.numpy()
    if len(np.unique(y)) < 2: return 0.5, 0.0
    skf = StratifiedKFold(
        n_splits=cv, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X[tr], y[tr])
        scores.append(roc_auc_score(
            y[te], clf.predict_proba(X[te])[:, 1]))
    return float(np.mean(scores)), float(np.std(scores))

print("Finding critical layer L* on TriviaQA...")
print("Sweeping all 28 layers on 20% selection set...\n")

# use 20% of train for layer selection
sel_data = TRAIN_DATA[:int(0.2 * len(TRAIN_DATA))]

layer_aurocs = {}
for layer_idx in range(NUM_LAYERS + 1):
    h, l = extract_hidden_states(sel_data, layer_idx, 1.0)
    layer_aurocs[layer_idx] = quick_auroc(h, l, cv=3)

TARGET_LAYER = max(layer_aurocs, key=layer_aurocs.get)
print(f"Layer AUROC summary (every 4 layers):")
for i in range(0, NUM_LAYERS + 1, 4):
    marker = " ← L*" if i == TARGET_LAYER else ""
    print(f"  Layer {i:2d}: {layer_aurocs[i]:.4f}{marker}")

print(f"\n★ Critical layer L* = {TARGET_LAYER} "
      f"({TARGET_LAYER/NUM_LAYERS*100:.0f}% network depth)")
print(f"  AUROC on selection set: {layer_aurocs[TARGET_LAYER]:.4f}")

# train final probe at L* on 60% train data
probe_train = TRAIN_DATA[int(0.2*len(TRAIN_DATA)):
                          int(0.8*len(TRAIN_DATA))]
h_tr, l_tr  = extract_hidden_states(probe_train, TARGET_LAYER, 1.0)
probe        = LogisticRegression(max_iter=1000, C=1.0)
probe.fit(h_tr.numpy(), l_tr.numpy())

# evaluate on held-out eval set
h_ev, l_ev  = extract_hidden_states(EVAL_DATA, TARGET_LAYER, 1.0)
proba        = probe.predict_proba(h_ev.numpy())[:, 1]
full_auroc   = roc_auc_score(l_ev.numpy(), proba)
_, full_std  = robust_auroc(h_ev, l_ev, cv=10)

print(f"\nProbe AUROC at L*={TARGET_LAYER} (full sequence):")
print(f"  {full_auroc:.4f} ± {full_std:.4f}")
print(f"\nProbe trained ✓  |  L* = {TARGET_LAYER}  |  "
      f"saved in variable TARGET_LAYER")

# save layer sweep results
with open(f'{SAVE_DIR}/layer_sweep.json', 'w') as f:
    json.dump({
        'target_layer': TARGET_LAYER,
        'layer_aurocs': layer_aurocs,
        'full_auroc':   full_auroc,
        'full_std':     full_std,
    }, f, indent=2)


### CELL 4 — Early detection sweep (10% to 100% tokens)

Tests detection at 10%, 20%, 30%, 50%, 100% token cutoffs.
At each cutoff we truncate the response to that fraction of
tokens before extracting the hidden state.
This answers: how early in generation can we detect a
hallucination using only the hidden state at L*?
L* is re-selected at each cutoff independently — some cutoffs
may have a different optimal layer than the full-sequence L*.

In [ ]:
# ── CELL 4 — Early detection sweep ───────────────────────────────
#
# Tests detection at 10%, 20%, 30%, 50%, 100% token cutoffs.
# At each cutoff we truncate the response to that fraction of
# tokens before extracting the hidden state.
#
# This answers: how early in generation can we detect a
# hallucination using only the hidden state at L*?
#
# L* is re-selected at each cutoff independently — some cutoffs
# may have a different optimal layer than the full-sequence L*.

CUTOFFS = [0.10, 0.20, 0.30, 0.50, 1.00]

print("Early detection sweep across token cutoffs...")
print(f"Using L*={TARGET_LAYER} found in Cell 3\n")

early_results = {}
print(f"{'Cutoff':>8} {'L*':>4} {'AUROC':>8} {'±Std':>8} "
      f"{'Signal':>12}")
print("-" * 50)

for cutoff in CUTOFFS:
    h_ev_c, l_ev_c = extract_hidden_states(
        EVAL_DATA, TARGET_LAYER, cutoff)
    mean, std = robust_auroc(h_ev_c, l_ev_c, cv=10)
    early_results[cutoff] = {
        'auroc': mean, 'std': std,
        'layer': TARGET_LAYER
    }
    if mean > 0.70:   signal = "strong ✓"
    elif mean > 0.60: signal = "reliable ~"
    elif mean > 0.55: signal = "weak"
    else:             signal = "chance ✗"

    print(f"{int(cutoff*100):>7}%   {TARGET_LAYER:>2}  "
          f"{mean:>8.4f}  ±{std:.4f}  {signal:>12}")

# find earliest reliable cutoff
earliest = None
for c in sorted(CUTOFFS):
    if early_results[c]['auroc'] > 0.60:
        earliest = c
        break

print(f"\nEarliest reliable detection: "
      f"{int(earliest*100) if earliest else 'none'}% tokens")
print(f"Full sequence AUROC: "
      f"{early_results[1.00]['auroc']:.4f}")

with open(f'{SAVE_DIR}/early_detection.json', 'w') as f:
    json.dump(early_results, f, indent=2)
print("Early detection results saved ✓")


### CELL 5 — RL activation steering — definitions

YOUR IDEA:
If we know a hidden state h at L* is pointing toward
hallucination, use RL to learn a steering vector Δh that
moves h toward the factual region of activation space.
HOW IT WORKS:
SteeringPolicy: h → Δh  (a small perturbation vector)
Steered state : h_new = h + Δh
REWARD for hallucinated samples:
how much closer is h_new to the factual centroid?
how much further is h_new from the hallu centroid?

In [ ]:
# ── CELL 5 — RL activation steering definitions ──────────────────
#
# YOUR IDEA:
#   If we know a hidden state h at L* is pointing toward
#   hallucination, use RL to learn a steering vector Δh that
#   moves h toward the factual region of activation space.
#
# HOW IT WORKS:
#   SteeringPolicy: h → Δh  (a small perturbation vector)
#   Steered state : h_new = h + Δh
#
#   REWARD for hallucinated samples:
#     how much closer is h_new to the factual centroid?
#     how much further is h_new from the hallu centroid?
#     → positive reward = we successfully steered toward factual
#
#   REWARD for factual samples:
#     how small is Δh? (don't disturb correct states)
#     → penalize large movements on already-correct activations
#
#   RL finds the optimal Δh for each hidden state through
#   trial and error, guided by the centroid reward signal.
#
# WHY THIS IS RL AND NOT SUPERVISED:
#   We don't know the "correct" Δh for each sample.
#   We only know whether the result was good (closer to
#   factual centroid) or bad (moved wrong direction).
#   RL learns from this scalar reward signal.
#
# WHAT STAYS FROZEN:
#   Llama-3.2-3B — never modified
#   Linear probe  — already trained, used for detection only
#
# WHAT GETS TRAINED:
#   SteeringPolicy — ~800K parameter MLP

class SteeringPolicy(nn.Module):
    """
    RL policy that learns to steer hallucinated activations
    toward the factual region of activation space.

    Architecture:
      Input  : h ∈ R^3072  (hidden state at L*)
      Hidden : 512 neurons
      Output : Δh ∈ R^3072 (steering vector, small magnitude)

    The steered state is h_new = h + Δh.

    scale parameter controls maximum perturbation magnitude.
    tanh ensures Δh stays bounded regardless of scale.
    """
    def __init__(self, hidden_dim=3072, scale=0.1):
        super().__init__()
        self.scale = scale
        self.net   = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, hidden_dim)
        )

    def forward(self, h):
        """Compute steering vector Δh."""
        delta = self.net(h)
        # tanh bounds direction, scale limits magnitude
        return torch.tanh(delta) * self.scale

    def steer(self, h):
        """Return steered state h + Δh."""
        return h + self.forward(h)


def compute_centroids(h_all, l_all):
    """
    Compute factual and hallucinated centroids.
    These are the targets for the steering policy:
      hallucinated states should move TOWARD factual centroid
      factual states should stay NEAR factual centroid
    """
    f_mask = l_all == 1
    h_mask = l_all == 0
    f_c = F.normalize(h_all[f_mask].mean(0), dim=-1)
    h_c = F.normalize(h_all[h_mask].mean(0), dim=-1)
    sep = F.cosine_similarity(
        f_c.unsqueeze(0), h_c.unsqueeze(0)).item()
    return f_c, h_c, sep


def compute_reward(h_orig, h_new, f_centroid,
                   h_centroid, true_label):
    """
    Reward function for the steering policy.

    For hallucinated samples (true_label=0):
      We WANT h_new to be closer to factual centroid than h_orig.
      reward = Δsim_factual + Δsim_away_from_hallu
      Positive reward = we moved in the right direction.
      Negative reward = we moved in the wrong direction.

    For factual samples (true_label=1):
      We do NOT want to disturb h.
      reward = -||Δh||² * penalty
      Penalizes large unnecessary movements.

    This reward is the RL objective:
      the policy maximizes expected reward across all samples,
      which means learning to push hallucinated states toward
      factual region without disturbing factual states.
    """
    h_o = h_orig.squeeze(0)
    h_n = h_new.squeeze(0)

    sim_o_f = F.cosine_similarity(
        h_o.unsqueeze(0), f_centroid.unsqueeze(0)).item()
    sim_n_f = F.cosine_similarity(
        h_n.unsqueeze(0), f_centroid.unsqueeze(0)).item()
    sim_o_h = F.cosine_similarity(
        h_o.unsqueeze(0), h_centroid.unsqueeze(0)).item()
    sim_n_h = F.cosine_similarity(
        h_n.unsqueeze(0), h_centroid.unsqueeze(0)).item()

    if true_label == 0:   # hallucinated → steer toward factual
        reward = (sim_n_f - sim_o_f) + (sim_o_h - sim_n_h)
    else:                  # factual → don't disturb
        movement = F.mse_loss(h_n, h_o).item()
        reward   = -movement * 10.0

    return reward, sim_o_f, sim_n_f


print("RL activation steering framework loaded ✓")
print(f"\nSteeringPolicy summary:")
total_params = sum(
    p.numel() for p in SteeringPolicy().parameters())
print(f"  Parameters  : {total_params:,}")
print(f"  Input dim   : 3072 (hidden state at L*={TARGET_LAYER})")
print(f"  Output dim  : 3072 (steering vector Δh)")
print(f"  Scale       : 0.1 (max perturbation = 10% of unit norm)")
print(f"\nReward logic:")
print(f"  Hallucinated h → positive reward if h+Δh closer to factual centroid")
print(f"  Factual h      → negative reward if Δh is large (don't disturb)")


### CELL 6 — Train RL steering policy (~15 min)

Trains the SteeringPolicy using gradient-based RL.
For each epoch, iterates over all training samples:
1. Extract h at L* (already done — using precomputed h_all)
2. Compute steering vector Δh via policy
3. Apply: h_new = h + Δh
4. Compute reward based on centroid proximity
5. Loss = -reward (maximize reward = minimize -reward)
6. Backprop through policy only (Llama frozen)
WHAT TO WATCH:
sep=  → centroid similarity, should DECREASE (clusters move apart)

In [ ]:
# ── CELL 6 — Train RL steering policy ────────────────────────────
#
# Trains the SteeringPolicy using gradient-based RL.
#
# For each epoch, iterates over all training samples:
#   1. Extract h at L* (already done — using precomputed h_all)
#   2. Compute steering vector Δh via policy
#   3. Apply: h_new = h + Δh
#   4. Compute reward based on centroid proximity
#   5. Loss = -reward (maximize reward = minimize -reward)
#   6. Backprop through policy only (Llama frozen)
#
# WHAT TO WATCH:
#   sep=  → centroid similarity, should DECREASE (clusters move apart)
#   AUROC → should INCREASE (steered states more separable)
#   hallu_rew → should be POSITIVE (hallucinated states moving correctly)
#   fact_rew  → should be close to 0 (factual states not disturbed)
#
# Expected runtime: ~15 minutes.

print("Pre-extracting all training hidden states at L*={TARGET_LAYER}...")
print("(done once, reused every epoch for speed)\n")

h_train_all, l_train_all = extract_hidden_states(
    TRAIN_DATA, TARGET_LAYER, 1.0)
h_train_dev = h_train_all.to(DEVICE)
l_train_dev = l_train_all.to(DEVICE)

# compute initial centroids
f_centroid, h_centroid, initial_sep = compute_centroids(
    h_train_dev, l_train_dev)

print(f"Training samples    : {len(TRAIN_DATA)}")
print(f"Factual             : {(l_train_dev==1).sum().item()}")
print(f"Hallucinated        : {(l_train_dev==0).sum().item()}")
print(f"Initial centroid similarity: {initial_sep:.4f}")
print(f"  (this should DECREASE during training)")

# initialize steering policy
hidden_dim = h_train_all.shape[1]
policy     = SteeringPolicy(hidden_dim=hidden_dim, scale=0.1).to(DEVICE)
optimizer  = torch.optim.Adam(policy.parameters(), lr=1e-3)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=30)

print(f"\nPolicy params : "
      f"{sum(p.numel() for p in policy.parameters()):,}")
print(f"Epochs        : 30")
print(f"Learning rate : 1e-3")
print(f"\nTraining...\n")

best_sep       = initial_sep
best_epoch     = 0
training_log   = []

for epoch in range(30):
    policy.train()
    hallu_rews, fact_rews = [], []
    idx = torch.randperm(len(h_train_dev))

    for i in idx:
        h          = h_train_dev[i].unsqueeze(0)
        true_label = l_train_dev[i].item()
        h_new      = policy.steer(h)

        reward, sim_o, sim_n = compute_reward(
            h, h_new,
            f_centroid, h_centroid, true_label)

        if true_label == 0:
            # differentiable: maximize similarity to factual centroid
            # and minimize similarity to hallu centroid
            loss = -(
                F.cosine_similarity(
                    F.normalize(h_new, dim=-1),
                    f_centroid.unsqueeze(0)) -
                F.cosine_similarity(
                    F.normalize(h_new, dim=-1),
                    h_centroid.unsqueeze(0))
            ).mean()
            hallu_rews.append(reward)
        else:
            # minimize movement for factual samples
            delta = policy(h)
            loss  = (delta ** 2).mean() * 10.0
            fact_rews.append(reward)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()

    scheduler.step()

    # evaluate every 3 epochs
    if (epoch + 1) % 3 == 0:
        policy.eval()
        with torch.no_grad():
            h_new_all = h_train_dev.clone()
            for i in range(len(h_train_dev)):
                if l_train_dev[i].item() == 0:
                    h_new_all[i] = policy.steer(
                        h_train_dev[i].unsqueeze(0)).squeeze(0)

            new_f_c = F.normalize(
                h_new_all[l_train_dev == 1].mean(0), dim=-1)
            new_h_c = F.normalize(
                h_new_all[l_train_dev == 0].mean(0), dim=-1)
            new_sep = F.cosine_similarity(
                new_f_c.unsqueeze(0),
                new_h_c.unsqueeze(0)).item()

            # quick AUROC on steered states
            from sklearn.linear_model import LogisticRegression as LR
            clf   = LR(max_iter=300)
            X_new = h_new_all.cpu().numpy()
            y_new = l_train_dev.cpu().numpy()
            clf.fit(X_new, y_new)
            proba_new = clf.predict_proba(X_new)[:, 1]
            auroc_new = roc_auc_score(y_new, proba_new)

        policy.train()

        h_r = np.mean(hallu_rews) if hallu_rews else 0
        f_r = np.mean(fact_rews)  if fact_rews  else 0
        print(f"  Epoch {epoch+1:3d}/30 | "
              f"sep={new_sep:.4f} "
              f"(init={initial_sep:.4f}) | "
              f"AUROC={auroc_new:.4f} | "
              f"hallu_rew={h_r:+.3f} | "
              f"fact_rew={f_r:+.3f}")

        training_log.append({
            'epoch':     epoch + 1,
            'sep':       new_sep,
            'auroc':     auroc_new,
            'hallu_rew': h_r,
            'fact_rew':  f_r,
        })

        if new_sep < best_sep:
            best_sep   = new_sep
            best_epoch = epoch + 1
            torch.save(policy.state_dict(),
                       f'{SAVE_DIR}/steering_policy_best.pt')

# save training log
with open(f'{SAVE_DIR}/training_log.json', 'w') as f:
    json.dump(training_log, f, indent=2)

print(f"\nTraining complete!")
print(f"Initial sep : {initial_sep:.4f}")
print(f"Best sep    : {best_sep:.4f} (epoch {best_epoch})")
print(f"Improvement : {initial_sep - best_sep:.4f}")
print(f"  (positive = clusters moved further apart ✓)")

# load best checkpoint
policy.load_state_dict(
    torch.load(f'{SAVE_DIR}/steering_policy_best.pt'))
policy.eval()
print(f"\nBest checkpoint loaded (epoch {best_epoch}) ✓")


### CELL 7 — Evaluate: before vs after steering

Compares the activation space BEFORE and AFTER steering.
METRICS:
Probe AUROC:
A linear probe trained on raw hidden states.
Measures how linearly separable factual vs hallu are.
Should INCREASE after steering (more separable).
Centroid cosine similarity:
Cosine similarity between factual and hallu centroids.
Should DECREASE after steering (more separated).
-1.0 = perfectly opposite. +1.0 = identical.

In [ ]:
# ── CELL 7 — Evaluate steering ────────────────────────────────────
#
# Compares the activation space BEFORE and AFTER steering.
#
# METRICS:
#
#   Probe AUROC:
#     A linear probe trained on raw hidden states.
#     Measures how linearly separable factual vs hallu are.
#     Should INCREASE after steering (more separable).
#
#   Centroid cosine similarity:
#     Cosine similarity between factual and hallu centroids.
#     Should DECREASE after steering (more separated).
#     -1.0 = perfectly opposite. +1.0 = identical.
#
#   Mean hallucination score per class:
#     For each sample: sim_to_hallu_centroid - sim_to_factual_centroid
#     Factual samples should be NEGATIVE (far from hallu centroid)
#     Hallu samples should be POSITIVE (close to hallu centroid)
#     After steering: hallu samples should move toward 0 or negative
#     because we pushed them toward the factual centroid.
#
#   Samples steered:
#     How many samples the probe flagged as hallucinated.
#     Only flagged samples get steered — factual ones unchanged.

print("Evaluating steering on held-out eval set...")
print(f"Eval samples: {len(EVAL_DATA)}\n")

h_eval_all, l_eval_all = extract_hidden_states(
    EVAL_DATA, TARGET_LAYER, 1.0)
h_eval_dev  = h_eval_all.to(DEVICE)
labels      = l_eval_all.numpy()

# apply steering — only to samples probe flags as hallucinated
h_steered   = h_eval_dev.clone()
n_steered   = 0
policy.eval()
with torch.no_grad():
    for i in range(len(h_eval_dev)):
        h_i   = h_eval_dev[i].unsqueeze(0)
        score = probe.predict_proba(h_i.cpu().numpy())[0, 1]
        if score > 0.5:   # probe says hallucinated → steer
            h_steered[i] = policy.steer(h_i).squeeze(0)
            n_steered    += 1

h_steered_cpu = h_steered.cpu()

# AUROC before and after
from sklearn.model_selection import cross_val_score as cvs
from sklearn.linear_model import LogisticRegression as LR

a_before = cvs(LR(max_iter=1000), h_eval_all.numpy(),
               labels, cv=5, scoring='roc_auc').mean()
a_after  = cvs(LR(max_iter=1000), h_steered_cpu.numpy(),
               labels, cv=5, scoring='roc_auc').mean()

# centroid separation before and after
fm = l_eval_all == 1
hm = l_eval_all == 0

fc_b  = F.normalize(h_eval_all[fm].mean(0), dim=-1)
hc_b  = F.normalize(h_eval_all[hm].mean(0), dim=-1)
sep_b = F.cosine_similarity(
    fc_b.unsqueeze(0), hc_b.unsqueeze(0)).item()

fc_a  = F.normalize(h_steered_cpu[fm].mean(0), dim=-1)
hc_a  = F.normalize(h_steered_cpu[hm].mean(0), dim=-1)
sep_a = F.cosine_similarity(
    fc_a.unsqueeze(0), hc_a.unsqueeze(0)).item()

# mean hallucination scores
scores_b = (h_eval_all @ hc_b - h_eval_all @ fc_b).numpy()
scores_a = (h_steered_cpu @ hc_a -
            h_steered_cpu @ fc_a).numpy()

# early detection: AUROC at each cutoff before vs after steering
print("Computing AUROC at each token cutoff (raw vs steered)...")
early_steer = {}
for cutoff in CUTOFFS:
    h_c, l_c = extract_hidden_states(EVAL_DATA, TARGET_LAYER, cutoff)
    h_c_dev  = h_c.to(DEVICE)

    # apply steering at this cutoff
    h_c_s = h_c_dev.clone()
    with torch.no_grad():
        for i in range(len(h_c_dev)):
            score = probe.predict_proba(
                h_c_dev[i:i+1].cpu().numpy())[0, 1]
            if score > 0.5:
                h_c_s[i] = policy.steer(
                    h_c_dev[i].unsqueeze(0)).squeeze(0)

    raw_a, _  = robust_auroc(h_c,          l_c, cv=5)
    steer_a,_ = robust_auroc(h_c_s.cpu(),  l_c, cv=5)
    early_steer[cutoff] = {
        'raw_auroc':   raw_a,
        'steer_auroc': steer_a,
        'delta':       steer_a - raw_a
    }

# print full results
print(f"\n{'='*60}")
print(f"  EVALUATION RESULTS")
print(f"{'='*60}")

print(f"\n── Main metrics (full sequence) ─────────────────────")
print(f"  {'Metric':<35} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"  {'-'*62}")
print(f"  {'Probe AUROC':<35} "
      f"{a_before:>8.4f} {a_after:>8.4f} "
      f"{a_after-a_before:>+8.4f}")
print(f"  {'Centroid cosine similarity':<35} "
      f"{sep_b:>8.4f} {sep_a:>8.4f} "
      f"{sep_a-sep_b:>+8.4f}")
print(f"  {'  (lower = more separated)':<35}")

print(f"\n── Hallucination scores per class ───────────────────")
print(f"  {'Class':<20} {'Before':>10} {'After':>10} {'Delta':>10}")
print(f"  {'-'*54}")
mhb = scores_b[labels == 0].mean()
mha = scores_a[labels == 0].mean()
mfb = scores_b[labels == 1].mean()
mfa = scores_a[labels == 1].mean()
print(f"  {'Hallucinated':<20} {mhb:>+10.4f} {mha:>+10.4f} "
      f"{mha-mhb:>+10.4f}")
print(f"  {'Factual':<20} {mfb:>+10.4f} {mfa:>+10.4f} "
      f"{mfa-mfb:>+10.4f}")
print(f"\n  Positive score = closer to hallu centroid")
print(f"  Negative score = closer to factual centroid")
print(f"  After steering: hallu score should DECREASE ↓")

print(f"\n── Early detection: raw vs steered ──────────────────")
print(f"  {'Cutoff':>8} {'Raw AUROC':>12} "
      f"{'Steered AUROC':>14} {'Delta':>8}")
print(f"  {'-'*48}")
for cutoff in CUTOFFS:
    r = early_steer[cutoff]
    print(f"  {int(cutoff*100):>7}%  {r['raw_auroc']:>12.4f}  "
          f"{r['steer_auroc']:>14.4f}  {r['delta']:>+8.4f}")

print(f"\n  Samples steered: {n_steered}/{len(EVAL_DATA)} "
      f"({100*n_steered/len(EVAL_DATA):.1f}%)")

# save results
eval_results = {
    'auroc_before':      a_before,
    'auroc_after':       a_after,
    'auroc_delta':       a_after - a_before,
    'sep_before':        sep_b,
    'sep_after':         sep_a,
    'sep_delta':         sep_a - sep_b,
    'mean_hallu_before': float(mhb),
    'mean_hallu_after':  float(mha),
    'mean_fact_before':  float(mfb),
    'mean_fact_after':   float(mfa),
    'n_steered':         n_steered,
    'n_eval':            len(EVAL_DATA),
    'early_steer':       early_steer,
}
with open(f'{SAVE_DIR}/eval_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

# interpretation
print(f"\n── Interpretation ────────────────────────────────────")
if sep_a < sep_b:
    print(f"  ✓ Centroid separation improved by {sep_b-sep_a:.4f}")
    print(f"    Hallucinated activations moved toward factual region")
else:
    print(f"  ~ Centroid separation did not improve")

if a_after > a_before:
    print(f"  ✓ AUROC improved by {a_after-a_before:+.4f}")
else:
    print(f"  ~ AUROC did not improve after steering")

if mha < mhb:
    print(f"  ✓ Hallucination score for hallu samples decreased")
    print(f"    ({mhb:+.4f} → {mha:+.4f}) — moved toward factual region")
else:
    print(f"  ~ Hallu score did not decrease")


### CELL 8 — Figures (5 plots) + final results table

Generates 5 figures saved to Google Drive:
Figure 1: Layer AUROC curve — shows where L* is
Figure 2: Early detection — AUROC vs token cutoff
Figure 3: PCA — activation space before vs after steering
Figure 4: Training log — sep and AUROC over epochs
Figure 5: Early detection raw vs steered comparison
Figure 1: layer AUROC curve

In [ ]:
# ── CELL 8 — Figures and final results ───────────────────────────
#
# Generates 5 figures saved to Google Drive:
#   Figure 1: Layer AUROC curve — shows where L* is
#   Figure 2: Early detection — AUROC vs token cutoff
#   Figure 3: PCA — activation space before vs after steering
#   Figure 4: Training log — sep and AUROC over epochs
#   Figure 5: Early detection raw vs steered comparison

# Figure 1: layer AUROC curve
fig, ax = plt.subplots(figsize=(10, 4))
layers     = list(range(NUM_LAYERS + 1))
auroc_vals = [layer_aurocs[i] for i in layers]
ax.plot(layers, auroc_vals, 'o-', color='steelblue',
        linewidth=1.5, markersize=4)
ax.axvline(TARGET_LAYER, color='coral', linestyle='--',
           linewidth=1.5, label=f'L*={TARGET_LAYER} '
           f'({TARGET_LAYER/NUM_LAYERS*100:.0f}% depth)')
ax.scatter([TARGET_LAYER], [layer_aurocs[TARGET_LAYER]],
           color='coral', s=120, zorder=5)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Layer depth', fontsize=11)
ax.set_ylabel('AUROC (linear probe)', fontsize=11)
ax.set_title(
    f'Figure 1: Hallucination detectability across all layers\n'
    f'TriviaQA — Llama-3.2-3B (frozen)',
    fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0.4, 1.0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig1_layer_sweep.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved ✓")

# Figure 2: early detection
fig, ax = plt.subplots(figsize=(8, 4))
xi      = range(len(CUTOFFS))
aurocs  = [early_results[c]['auroc'] for c in CUTOFFS]
stds    = [early_results[c]['std']   for c in CUTOFFS]
ax.plot(xi, aurocs, 'o-', color='steelblue',
        linewidth=2, markersize=8)
ax.fill_between(xi,
    [a-s for a, s in zip(aurocs, stds)],
    [a+s for a, s in zip(aurocs, stds)],
    alpha=0.15, color='steelblue')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5,
           label='Chance (0.5)')
ax.axhline(0.65, color='green', linestyle='--', alpha=0.5,
           label='Reliable threshold (0.65)')
ax.set_xticks(xi)
ax.set_xticklabels([f"{int(c*100)}%" for c in CUTOFFS],
                   fontsize=11)
ax.set_xlabel('Tokens generated (% of full response)',
              fontsize=11)
ax.set_ylabel('AUROC', fontsize=11)
ax.set_title(
    f'Figure 2: Early hallucination detection at L*={TARGET_LAYER}\n'
    f'How early can we detect? (TriviaQA)',
    fontsize=11)
ax.set_ylim(0.4, 1.0)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig2_early_detection.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved ✓")

# Figure 3: PCA before vs after steering
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
y_plot    = labels

for ax, h_plot, title in [
    (axes[0], h_eval_all,     f'Before steering (L*={TARGET_LAYER})'),
    (axes[1], h_steered_cpu,  'After RL activation steering')
]:
    reduced = PCA(n_components=2).fit_transform(h_plot.numpy())
    fm_p    = y_plot == 1
    ax.scatter(reduced[fm_p,  0], reduced[fm_p,  1],
               c='steelblue', alpha=0.6, s=20, label='Factual')
    ax.scatter(reduced[~fm_p, 0], reduced[~fm_p, 1],
               c='coral',     alpha=0.6, s=20, label='Hallucinated')
    h_t = torch.tensor(h_plot.numpy())
    fc  = F.normalize(h_t[fm_p].mean(0), dim=-1)
    hc  = F.normalize(h_t[~fm_p].mean(0), dim=-1)
    sep = F.cosine_similarity(fc.unsqueeze(0), hc.unsqueeze(0)).item()
    ax.set_title(f'Figure 3: {title}\nCentroid sim={sep:.4f}',
                 fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle(
    'Activation space at L* before and after RL steering\n'
    'Lower centroid similarity = better separation',
    fontsize=12)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig3_pca_before_after.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved ✓")

# Figure 4: training log
if training_log:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs_log = [t['epoch']   for t in training_log]
    seps       = [t['sep']     for t in training_log]
    aurocs_log = [t['auroc']   for t in training_log]
    h_rews     = [t['hallu_rew'] for t in training_log]

    ax = axes[0]
    ax.plot(epochs_log, seps, 'o-', color='coral',
            linewidth=1.5, markersize=6)
    ax.axhline(initial_sep, color='gray', linestyle='--',
               alpha=0.5, label=f'Initial ({initial_sep:.4f})')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Centroid cosine similarity', fontsize=11)
    ax.set_title('Figure 4a: Centroid separation during training\n'
                 '(decreasing = clusters moving apart)',
                 fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(epochs_log, aurocs_log, 'o-', color='steelblue',
            linewidth=1.5, markersize=6, label='AUROC')
    ax.plot(epochs_log, h_rews, 's--', color='coral',
            linewidth=1.5, markersize=6, label='Hallu reward')
    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_title('Figure 4b: AUROC and hallu reward during training',
                 fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/fig4_training_log.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure 4 saved ✓")

# Figure 5: early detection raw vs steered
fig, ax = plt.subplots(figsize=(8, 5))
xi         = range(len(CUTOFFS))
raw_aurocs = [early_steer[c]['raw_auroc']   for c in CUTOFFS]
steer_aurocs = [early_steer[c]['steer_auroc'] for c in CUTOFFS]
ax.plot(xi, raw_aurocs,   'o-', color='steelblue',
        linewidth=2, markersize=8, label='Raw hidden states')
ax.plot(xi, steer_aurocs, 's-', color='coral',
        linewidth=2, markersize=8, label='After RL steering')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(xi)
ax.set_xticklabels([f"{int(c*100)}%" for c in CUTOFFS],
                   fontsize=11)
ax.set_xlabel('Tokens generated', fontsize=11)
ax.set_ylabel('AUROC', fontsize=11)
ax.set_title(
    f'Figure 5: Early detection — raw vs RL steered\n'
    f'Layer L*={TARGET_LAYER} — TriviaQA',
    fontsize=11)
ax.set_ylim(0.4, 1.0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig5_early_detection_steered.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 5 saved ✓")

# Final results table
print(f"\n{'='*60}")
print(f"  FINAL RESULTS — TriviaQA + RL Activation Steering")
print(f"{'='*60}")
print(f"\nModel  : Llama-3.2-3B-Instruct (frozen throughout)")
print(f"Dataset: TriviaQA (2000 samples, 80/20 split)")

print(f"\n── Critical layer (Cell 3) ───────────────────────────")
print(f"  L* = {TARGET_LAYER} ({TARGET_LAYER/NUM_LAYERS*100:.0f}% network depth)")
print(f"  AUROC at L* (full seq): {full_auroc:.4f} ± {full_std:.4f}")

print(f"\n── Early detection (Cell 4) ──────────────────────────")
print(f"  {'Cutoff':>8} {'AUROC':>8} {'±Std':>8}")
print(f"  {'-'*28}")
for cutoff in CUTOFFS:
    r = early_results[cutoff]
    print(f"  {int(cutoff*100):>7}%  {r['auroc']:>8.4f}  "
          f"±{r['std']:.4f}")

print(f"\n── RL activation steering (Cell 6-7) ────────────────")
print(f"  {'Metric':<30} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"  {'-'*58}")
print(f"  {'AUROC (full sequence)':<30} "
      f"{a_before:>8.4f} {a_after:>8.4f} "
      f"{a_after-a_before:>+8.4f}")
print(f"  {'Centroid similarity':<30} "
      f"{sep_b:>8.4f} {sep_a:>8.4f} "
      f"{sep_a-sep_b:>+8.4f}")
print(f"  {'Hallu score (hallu samples)':<30} "
      f"{mhb:>+8.4f} {mha:>+8.4f} "
      f"{mha-mhb:>+8.4f}")
print(f"\n  Samples steered: {n_steered}/{len(EVAL_DATA)}")

print(f"\n── Early detection raw vs steered ────────────────────")
print(f"  {'Cutoff':>8} {'Raw':>8} {'Steered':>10} {'Delta':>8}")
print(f"  {'-'*40}")
for cutoff in CUTOFFS:
    r = early_steer[cutoff]
    print(f"  {int(cutoff*100):>7}%  {r['raw_auroc']:>8.4f}  "
          f"{r['steer_auroc']:>10.4f}  {r['delta']:>+8.4f}")

print(f"\n── Files saved to {SAVE_DIR}/ ──────────")
for fname in [
    'labeled_data.json',
    'layer_sweep.json',
    'early_detection.json',
    'training_log.json',
    'eval_results.json',
    'steering_policy_best.pt',
    'fig1_layer_sweep.png',
    'fig2_early_detection.png',
    'fig3_pca_before_after.png',
    'fig4_training_log.png',
    'fig5_early_detection_steered.png',
]:
    path = f'{SAVE_DIR}/{fname}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"  {fname:<40} {size:>8.1f} KB ✓")
    else:
        print(f"  {fname:<40} missing ✗")

print(f"\n{'='*60}")
print(f"  DONE")
print(f"{'='*60}")


In [ ]:
# ── CELL 7 — Evaluate label-free steering ────────────────────────
# Steering decision at eval time also uses probe only.
# TriviaQA labels used ONLY to compute final AUROC metric
# (you need them to measure performance — but not to steer).

print("Evaluating label-free steering on held-out eval set...")

h_eval_all, l_eval_all = extract_hidden_states(
    EVAL_DATA, TARGET_LAYER, 1.0)
h_eval_dev = h_eval_all.to(DEVICE)
labels     = l_eval_all.numpy()

# probe scores on eval — no labels used
eval_probe_scores = probe.predict_proba(
    h_eval_all.numpy())[:, 1]

# steer samples where probe score > 0.5 (no label used)
h_steered   = h_eval_dev.clone()
n_steered   = 0
policy.eval()
with torch.no_grad():
    for i in range(len(h_eval_dev)):
        if eval_probe_scores[i] > 0.5:
            h_steered[i] = policy.steer(
                h_eval_dev[i].unsqueeze(0)).squeeze(0)
            n_steered += 1

h_steered_cpu = h_steered.cpu()

# AUROC before and after (labels used only here for measurement)
from sklearn.model_selection import cross_val_score as cvs
from sklearn.linear_model import LogisticRegression as LR

a_before = cvs(LR(max_iter=1000), h_eval_all.numpy(),
               labels, cv=5, scoring='roc_auc').mean()
a_after  = cvs(LR(max_iter=1000), h_steered_cpu.numpy(),
               labels, cv=5, scoring='roc_auc').mean()

# centroid separation
fm = l_eval_all == 1
hm = l_eval_all == 0
fc_b  = F.normalize(h_eval_all[fm].mean(0), dim=-1)
hc_b  = F.normalize(h_eval_all[hm].mean(0), dim=-1)
sep_b = F.cosine_similarity(
    fc_b.unsqueeze(0), hc_b.unsqueeze(0)).item()
fc_a  = F.normalize(h_steered_cpu[fm].mean(0), dim=-1)
hc_a  = F.normalize(h_steered_cpu[hm].mean(0), dim=-1)
sep_a = F.cosine_similarity(
    fc_a.unsqueeze(0), hc_a.unsqueeze(0)).item()

# early detection
early_steer_lf = {}
for cutoff in CUTOFFS:
    h_c, l_c = extract_hidden_states(
        EVAL_DATA, TARGET_LAYER, cutoff)
    h_c_dev  = h_c.to(DEVICE)
    scores_c = probe.predict_proba(h_c.numpy())[:, 1]

    h_c_s = h_c_dev.clone()
    with torch.no_grad():
        for i in range(len(h_c_dev)):
            if scores_c[i] > 0.5:
                h_c_s[i] = policy.steer(
                    h_c_dev[i].unsqueeze(0)).squeeze(0)

    raw_a, _   = robust_auroc(h_c,        l_c, cv=5)
    steer_a, _ = robust_auroc(h_c_s.cpu(),l_c, cv=5)
    early_steer_lf[cutoff] = {
        'raw_auroc':   raw_a,
        'steer_auroc': steer_a,
        'delta':       steer_a - raw_a
    }

# print results
print(f"\n{'='*60}")
print(f"  LABEL-FREE RL STEERING — EVALUATION RESULTS")
print(f"  (Zero TriviaQA labels used in RL training or steering)")
print(f"{'='*60}")

print(f"\n── Main metrics (full sequence) ─────────────────────")
print(f"  {'Metric':<35} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"  {'-'*62}")
print(f"  {'Probe AUROC':<35} "
      f"{a_before:>8.4f} {a_after:>8.4f} "
      f"{a_after-a_before:>+8.4f}")
print(f"  {'Centroid cosine similarity':<35} "
      f"{sep_b:>8.4f} {sep_a:>8.4f} "
      f"{sep_a-sep_b:>+8.4f}")

print(f"\n── Early detection raw vs steered ────────────────────")
print(f"  {'Cutoff':>8} {'Raw':>8} {'Steered':>10} {'Delta':>8}")
print(f"  {'-'*40}")
for cutoff in CUTOFFS:
    r = early_steer_lf[cutoff]
    print(f"  {int(cutoff*100):>7}%  {r['raw_auroc']:>8.4f}  "
          f"{r['steer_auroc']:>10.4f}  {r['delta']:>+8.4f}")

print(f"\n  Samples steered: {n_steered}/{len(EVAL_DATA)} "
      f"({100*n_steered/len(EVAL_DATA):.1f}%)")

print(f"\n── What the RL used vs what it did not use ───────────")
print(f"  Used    : probe score P(hallucinated) for steering decision")
print(f"  Used    : probe-derived centroids for reward signal")
print(f"  NOT used: TriviaQA gold answers during RL training")
print(f"  NOT used: TriviaQA labels during steering at eval time")
print(f"  Labels used ONLY to compute final AUROC metric")